# Pertemuan 6 — Pencarian Adversarial: Minimax dan Alpha-Beta Pruning
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Sampai pertemuan 5 agen bekerja sendirian. Dalam **permainan**, ada lawan yang secara aktif berusaha menggagalkan rencana kita. Solusi bukan lagi urutan aksi, melainkan **strategi**: langkah terbaik untuk setiap kemungkinan balasan lawan.

**Setelah menyelesaikan notebook ini Anda mampu:**
1. memformulasikan permainan dua pemain sebagai masalah pencarian adversarial;
2. mengimplementasikan minimax dan alpha-beta pruning serta mengukur penghematannya;
3. memakai batas kedalaman dan fungsi evaluasi ketika pohon permainan terlalu besar.

Notebook ini adalah tahap pertama asesmen **A3** (formatif). Kerjakan berpasangan (*pair programming*): satu orang mengetik, satu orang menelaah, bertukar peran tiap bagian.

In [ ]:
NAMA = "tulis nama Anda dan pasangan"
NIM = "tulis NIM Anda dan pasangan"
print(f"Notebook P6 dikerjakan oleh: {NAMA} ({NIM})")

import math, random, time
import pandas as pd
import matplotlib.pyplot as plt

## 1. Minimax pada pohon permainan kecil

Dua pemain: **MAX** (kita) ingin nilai sebesar mungkin, **MIN** (lawan) ingin nilai sekecil mungkin. Keduanya diasumsikan bermain optimal.

Pohon contoh di slide: MAX memilih satu dari tiga langkah, lalu MIN memilih satu dari tiga balasan. Angka pada daun adalah nilai utilitas bagi MAX. Pohon ditulis sebagai *list* bersarang.

In [ ]:
POHON = [[3, 12, 8], [2, 4, 6], [14, 5, 2]]

def minimax_pohon(simpul, giliran_max, tingkat=0, jejak=False):
    if not isinstance(simpul, list):                      # daun: kembalikan utilitasnya
        return simpul
    nilai_anak = [minimax_pohon(anak, not giliran_max, tingkat + 1, jejak) for anak in simpul]
    nilai = max(nilai_anak) if giliran_max else min(nilai_anak)
    if jejak:
        print("   " * tingkat + f"{'MAX' if giliran_max else 'MIN'} memilih {nilai} dari {nilai_anak}")
    return nilai

print("Nilai minimax akar:", minimax_pohon(POHON, True, jejak=True))

MAX memilih langkah pertama karena **jaminan terburuknya** paling baik (3). Langkah ketiga memang memuat daun 14, tetapi lawan yang cerdas tidak akan pernah membiarkan kita mendapatkannya.

## 2. Alpha-beta pruning
Minimax memeriksa semua daun. Banyak di antaranya sebenarnya tidak perlu dilihat. Alpha-beta membawa dua batas selama penelusuran:

- **α (alpha)**: nilai terbaik yang sudah dijamin untuk MAX di sepanjang jalur ini;
- **β (beta)**: nilai terbaik yang sudah dijamin untuk MIN di sepanjang jalur ini.

Begitu `α ≥ β`, sisa cabang **dipangkas**: lawan tidak akan pernah membiarkan permainan masuk ke cabang itu. Hasil akhirnya **sama persis** dengan minimax.

In [ ]:
def alphabeta_pohon(simpul, giliran_max, alpha=-math.inf, beta=math.inf, catatan=None, lintasan="akar"):
    if not isinstance(simpul, list):
        catatan["daun"].append(simpul)
        return simpul
    nilai = -math.inf if giliran_max else math.inf
    for i, anak in enumerate(simpul):
        v = alphabeta_pohon(anak, not giliran_max, alpha, beta, catatan, f"{lintasan}.{i + 1}")
        if giliran_max:
            nilai = max(nilai, v); alpha = max(alpha, nilai)
        else:
            nilai = min(nilai, v); beta = min(beta, nilai)
        if alpha >= beta:
            sisa = len(simpul) - i - 1
            if sisa:
                catatan["pangkas"].append(f"di {lintasan}: {sisa} cabang dipangkas (α = {alpha}, β = {beta})")
            break
    return nilai

catatan = {"daun": [], "pangkas": []}
print("Nilai akar       :", alphabeta_pohon(POHON, True, catatan=catatan))
print("Daun yang dilihat:", catatan["daun"], f"({len(catatan['daun'])} dari 9)")
for p in catatan["pangkas"]: print("Pemangkasan      :", p)

Pada cabang kedua, setelah melihat daun `2`, MIN sudah bisa menjamin nilai ≤ 2. Padahal MAX sudah memegang jaminan 3 dari cabang pertama. Cabang kedua pasti tidak dipilih, jadi daun `4` dan `6` tidak perlu dibuka.

### Urutan langkah menentukan banyaknya pemangkasan

In [ ]:
def hitung_daun(pohon):
    c = {"daun": [], "pangkas": []}; alphabeta_pohon(pohon, True, catatan=c); return len(c["daun"])

urutan_baik  = [[3, 8, 12], [2, 4, 6], [2, 5, 14]]       # langkah terbaik diperiksa lebih dulu
urutan_buruk = [[2, 5, 14], [6, 4, 2], [12, 8, 3]]       # langkah terbaik diperiksa paling akhir
print("Urutan asli :", hitung_daun(POHON), "daun diperiksa")
print("Urutan baik :", hitung_daun(urutan_baik), "daun diperiksa")
print("Urutan buruk:", hitung_daun(urutan_buruk), "daun diperiksa")

Pohon yang sama, nilai akar yang sama, tetapi jumlah daun yang diperiksa berbeda jauh. Dengan urutan langkah ideal, alpha-beta hanya memeriksa sekitar O(b^(d/2)) node: **kedalaman pencarian bisa dua kali lipat** dengan waktu yang sama.

## 3. Formulasi permainan: tic-tac-toe
Enam komponen formulasi permainan:

| Komponen | Tic-tac-toe |
|---|---|
| Status awal | papan kosong |
| `giliran(s)` | X bila jumlah X = jumlah O, selain itu O |
| `aksi(s)` | petak kosong (0–8) |
| `hasil(s, a)` | papan setelah pemain yang giliran mengisi petak `a` |
| `terminal(s)` | ada tiga segaris, atau papan penuh |
| `utilitas(s)` | +1 bila X menang, −1 bila O menang, 0 bila seri |

Papan disimpan sebagai tuple 9 elemen berisi `"X"`, `"O"`, atau `None`.

In [ ]:
GARIS = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]
KOSONG = (None,) * 9

def giliran(s):
    return "X" if s.count("X") == s.count("O") else "O"

def aksi(s):
    return [i for i in range(9) if s[i] is None]

def hasil(s, a):
    papan = list(s); papan[a] = giliran(s); return tuple(papan)

def pemenang(s):
    for i, j, k in GARIS:
        if s[i] is not None and s[i] == s[j] == s[k]:
            return s[i]
    return None

def terminal(s):
    return pemenang(s) is not None or all(x is not None for x in s)

def utilitas(s):
    return {"X": 1, "O": -1, None: 0}[pemenang(s)]

def cetak(s):
    for r in range(3):
        print(" " + " | ".join(s[3 * r + c] or str(3 * r + c) for c in range(3)))
        if r < 2: print("---+---+---")

s = hasil(hasil(hasil(KOSONG, 4), 0), 8)
cetak(s); print("\nGiliran:", giliran(s), "| aksi:", aksi(s), "| terminal:", terminal(s))

In [ ]:
penghitung = {"node": 0}

def minimax(s):
    """Mengembalikan (nilai, langkah terbaik) bagi pemain yang sedang giliran."""
    penghitung["node"] += 1
    if terminal(s):
        return utilitas(s), None
    max_main = giliran(s) == "X"
    terbaik, langkah = (-math.inf if max_main else math.inf), None
    for a in aksi(s):
        v, _ = minimax(hasil(s, a))
        if (max_main and v > terbaik) or (not max_main and v < terbaik):
            terbaik, langkah = v, a
    return terbaik, langkah

def alphabeta(s, alpha=-math.inf, beta=math.inf):
    penghitung["node"] += 1
    if terminal(s):
        return utilitas(s), None
    max_main = giliran(s) == "X"
    terbaik, langkah = (-math.inf if max_main else math.inf), None
    for a in aksi(s):
        v, _ = alphabeta(hasil(s, a), alpha, beta)
        if max_main:
            if v > terbaik: terbaik, langkah = v, a
            alpha = max(alpha, terbaik)
        else:
            if v < terbaik: terbaik, langkah = v, a
            beta = min(beta, terbaik)
        if alpha >= beta:
            break
    return terbaik, langkah

ukur = []
for nama, fungsi in (("Minimax", minimax), ("Alpha-beta", alphabeta)):
    penghitung["node"] = 0; t = time.time()
    nilai, langkah = fungsi(KOSONG)
    ukur.append({"algoritma": nama, "nilai papan kosong": nilai, "langkah pertama": langkah,
                 "node diperiksa": penghitung["node"], "waktu (detik)": round(time.time() - t, 2)})
tabel_ukur = pd.DataFrame(ukur)
print(tabel_ukur.to_string(index=False))

**Dua temuan penting:**
1. Nilai papan kosong adalah **0**. Bila kedua pemain bermain sempurna, tic-tac-toe selalu seri.
2. Alpha-beta memberi jawaban yang sama dengan memeriksa hanya sebagian kecil node.

In [ ]:
plt.figure(figsize=(6.5, 3.6))
plt.bar(tabel_ukur["algoritma"], tabel_ukur["node diperiksa"], color=["#8A9AA5", "#0E7C86"])
for i, v in enumerate(tabel_ukur["node diperiksa"]):
    plt.text(i, v, f"{v:,}".replace(",", "."), ha="center", va="bottom")
plt.ylabel("node diperiksa"); plt.title("Tic-tac-toe dari papan kosong"); plt.show()

### Bermain melawan AI
Fungsi `main_lawan_ai()` memungkinkan Anda bermain sebagai O melawan AI (X). Hapus tanda `#` pada baris terakhir untuk mencobanya. Mungkinkah Anda menang?

In [ ]:
def langkah_ai(s):
    return alphabeta(s)[1]

def main_lawan_ai():
    s = KOSONG
    while not terminal(s):
        if giliran(s) == "X":
            s = hasil(s, langkah_ai(s))
        else:
            cetak(s)
            a = int(input("Petak pilihan Anda (0-8): "))
            if a not in aksi(s):
                print("Petak tidak sah."); continue
            s = hasil(s, a)
    cetak(s); print("Hasil:", pemenang(s) or "seri")

# main_lawan_ai()

In [ ]:
def main_otomatis(pemain_x, pemain_o):
    s = KOSONG
    while not terminal(s):
        s = hasil(s, (pemain_x if giliran(s) == "X" else pemain_o)(s))
    return pemenang(s) or "seri"

acak = random.Random(0)
def pemain_acak(s): return acak.choice(aksi(s))

rekap = pd.Series([main_otomatis(langkah_ai, pemain_acak) for _ in range(30)]).value_counts()
print("AI (X) melawan pemain acak (O), 30 pertandingan:"); print(rekap.to_string())

## 4. Batas kedalaman dan fungsi evaluasi
Catur memiliki sekitar 35 langkah sah per giliran dan permainan berlangsung puluhan giliran. Pohonnya mustahil ditelusuri sampai akhir. Solusinya:

1. **hentikan** pencarian pada kedalaman tertentu (*cutoff*);
2. nilai status non-terminal dengan **fungsi evaluasi**: taksiran peluang menang berdasarkan ciri-ciri papan.

Untuk tic-tac-toe kita gunakan evaluasi sederhana: *(jumlah garis yang masih terbuka bagi X) − (jumlah garis yang masih terbuka bagi O)*, dibagi 10 agar tetap berada di antara −1 dan +1.

In [ ]:
def evaluasi(s):
    if terminal(s): return utilitas(s)
    terbuka_x = sum(1 for g in GARIS if all(s[i] != "O" for i in g))
    terbuka_o = sum(1 for g in GARIS if all(s[i] != "X" for i in g))
    return (terbuka_x - terbuka_o) / 10

def alphabeta_batas(s, kedalaman, alpha=-math.inf, beta=math.inf):
    penghitung["node"] += 1
    if terminal(s) or kedalaman == 0:
        return evaluasi(s), None
    max_main = giliran(s) == "X"
    terbaik, langkah = (-math.inf if max_main else math.inf), None
    for a in aksi(s):
        v, _ = alphabeta_batas(hasil(s, a), kedalaman - 1, alpha, beta)
        if max_main:
            if v > terbaik: terbaik, langkah = v, a
            alpha = max(alpha, terbaik)
        else:
            if v < terbaik: terbaik, langkah = v, a
            beta = min(beta, terbaik)
        if alpha >= beta: break
    return terbaik, langkah

baris = []
for d in (1, 2, 3, 4, 6, 9):
    penghitung["node"] = 0
    alphabeta_batas(KOSONG, d); node_awal = penghitung["node"]
    acak = random.Random(1)
    # AI berbatas kedalaman bermain sebagai O melawan AI sempurna (X)
    hasil_main = pd.Series([main_otomatis(lambda s: acak.choice(aksi(s)) if s == KOSONG else langkah_ai(s),
                                          lambda s, d=d: alphabeta_batas(s, d)[1]) for _ in range(20)]).value_counts()
    baris.append({"batas kedalaman": d, "node (langkah pertama)": node_awal,
                  "seri": int(hasil_main.get("seri", 0)), "kalah": int(hasil_main.get("X", 0))})
print("AI berbatas kedalaman (O) melawan AI sempurna (X), 20 pertandingan dengan langkah pembuka X acak:")
print(pd.DataFrame(baris).to_string(index=False))

Secara umum, makin dalam pencarian makin jarang AI kalah, tetapi makin banyak node yang harus diperiksa. Pada kedalaman dangkal hasilnya sangat bergantung pada fungsi evaluasi. Mutu **fungsi evaluasi** menentukan seberapa dangkal pencarian boleh dilakukan tanpa mengorbankan mutu permainan.

---
## Latihan (tahap 1 asesmen A3, formatif)

### Latihan 1 — Permainan Nim (ambil batang korek)
Aturan: terdapat `n` batang korek. Dua pemain bergiliran mengambil **1, 2, atau 3** batang. Pemain yang mengambil batang **terakhir menang**.

Lengkapi formulasi berikut. Status: tuple `(sisa_batang, pemain_giliran)` dengan pemain `"MAX"` atau `"MIN"`.

In [ ]:
def nim_aksi(s):
    # TODO: kembalikan daftar jumlah batang yang boleh diambil
    return []

def nim_hasil(s, a):
    # TODO: kembalikan status setelah pemain yang giliran mengambil a batang
    return s

def nim_terminal(s):
    # TODO: permainan selesai bila batang habis
    return True

def nim_utilitas(s):
    # TODO: +1 bila MAX yang mengambil batang terakhir, -1 bila MIN
    return 0

def nim_minimax(s):
    if nim_terminal(s):
        return nim_utilitas(s), None
    max_main = s[1] == "MAX"
    terbaik, langkah = (-math.inf if max_main else math.inf), None
    for a in nim_aksi(s):
        v, _ = nim_minimax(nim_hasil(s, a))
        if (max_main and v > terbaik) or (not max_main and v < terbaik):
            terbaik, langkah = v, a
    return terbaik, langkah

if nim_aksi((5, "MAX")):
    for n in range(1, 13):
        nilai, langkah = nim_minimax((n, "MAX"))
        print(f"n = {n:2d}: {'MAX menang' if nilai > 0 else 'MAX kalah '} | langkah terbaik: ambil {langkah}")
else:
    print("Formulasi Nim belum dilengkapi.")

**Pertanyaan:** dari tabel hasil, pola apa yang terlihat? Untuk `n` berapa saja pemain pertama pasti kalah? Rumuskan strategi menangnya dalam satu kalimat.

*Jawaban:*

### Latihan 2 — Alpha-beta untuk Nim
Tulis `nim_alphabeta` lalu bandingkan jumlah node yang diperiksa dengan `nim_minimax` untuk `n = 10, 15, 20`. Sajikan dalam tabel.

In [ ]:
# Jawaban Latihan 2

### Latihan 3 — Memperbaiki fungsi evaluasi tic-tac-toe
Rancang fungsi evaluasi yang lebih baik daripada `evaluasi` (misalnya memberi bobot lebih untuk garis yang sudah berisi dua tanda sendiri). Uji dengan eksperimen yang sama seperti bagian 4 pada batas kedalaman 1 dan 2. Apakah jumlah kekalahan berkurang?

In [ ]:
# Jawaban Latihan 3

### Refleksi
1. Minimax mengasumsikan lawan bermain optimal. Apa yang terjadi bila lawan ternyata bermain buruk: apakah keputusan minimax menjadi salah?
2. Mengapa alpha-beta tidak pernah mengubah nilai akar?
3. Sebutkan satu situasi nyata di luar permainan papan yang dapat dimodelkan sebagai pencarian adversarial, dan satu risiko etis dari penerapannya.

*Jawaban:*

---
**Pengumpulan:** notebook ini dikumpulkan sebagai bukti tahap A3 (formatif) lewat LMS dengan nama `P06_NIM1_NIM2.ipynb`.